In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
df_order_items=spark.sql("select * from workspace.silver_olist.silver_order_items")

In [0]:
df_order_items.printSchema()

In [0]:
dimCustomers=spark.sql("select * from gold_olist.dimcustomers")
dimProducts=spark.sql("select * from gold_olist.dimproducts")
dimsellers=spark.sql("select * from gold_olist.dimsellers")
dimPayments=spark.sql("select * from gold_olist.dimpayments")
dimDate=spark.sql("select * from gold_olist.dimdate")
dimOrders=spark.sql("select * from workspace.silver_olist.silver_orders")

In [0]:
fact_order_items = (
    df_order_items
    .join(
        dimOrders.select(
            "order_id",
            "customer_id",
            "order_purchase_timestamp"
        ),
        "order_id",
        "left"
    )
    .join(
        dimCustomers.select("DimCustomerKey", "customer_id"),
        "customer_id",
        "left"
    )
    .join(
        dimProducts.select("DimproductKey", "product_id"),
        "product_id",
        "left"
    )
    .join(
        dimsellers.select("DimSellerKey", "seller_id"),
        "seller_id",
        "left"
    )
    .join(
        dimDate.select("DateKey", "full_date"),
        col("order_purchase_timestamp").cast("date") == dimDate.full_date,
        "left"
    )
    .select(
        col("order_id"),
        col("order_item_id"),
        col("DimCustomerKey"),
        col("DimproductKey"),
        col("DimSellerKey"),
        col("DateKey").alias("DimDateKey"),
        col("price"),
        col("freight_value"),
        (col("price") + col("freight_value")).alias("total_order_value"),
        col("shipping_limit_date")
    )
)
fact_order_items.show(n=10)

In [0]:
display(dimOrders.select("order_id", "customer_id").limit(5))

display(dimCustomers.select("customer_id", "DimCustomerKey").limit(5))